In [1]:
from bs4 import BeautifulSoup
import requests
import re

op_codes_url = 'https://www.masswerk.at/6502/6502_instruction_set.html#ASL'

def fetch_soup(url):
    response = requests.get(url)
    return BeautifulSoup(response.text, 'html.parser')
op_codes_soup = fetch_soup(op_codes_url)

In [2]:
op_codes_soup = fetch_soup(op_codes_url)

In [3]:
# get the table of opcodes and translate it into a matrix
import pandas as pd
from IPython.display import Markdown
op_codes_table = op_codes_soup.find('table')
op_codes = []
for row in op_codes_table.find_all('tr'):
    cols = row.find_all('td')
    if len(cols) == 0:
        continue
    op_codes.append([col.text for col in cols])

# remove the header row
op_codes = op_codes

op_codes_df = pd.DataFrame(op_codes)
# convert decimal to hex for the index column and the header row
op_codes_df = op_codes_df.replace('---', None)
op_codes_df.columns = [hex(int(col)) for col in op_codes_df.columns]
op_codes_df[''] = op_codes_df.columns
op_codes_df.reset_index(drop=True, inplace=True)
op_codes_df.set_index('', inplace=True)
op_codes_df.columns.name = ''
op_codes_df.index.name = 'V hi/lo >'
op_codes_md = op_codes_df.to_markdown()
Markdown(op_codes_md)

| V hi/lo >   | 0x0      | 0x1       | 0x2   | 0x3   | 0x4       | 0x5       | 0x6       | 0x7   | 0x8      | 0x9       | 0xa      | 0xb   | 0xc       | 0xd       | 0xe       | 0xf   |
|:------------|:---------|:----------|:------|:------|:----------|:----------|:----------|:------|:---------|:----------|:---------|:------|:----------|:----------|:----------|:------|
| 0x0         | BRK impl | ORA X,ind |       |       |           | ORA zpg   | ASL zpg   |       | PHP impl | ORA #     | ASL A    |       |           | ORA abs   | ASL abs   |       |
| 0x1         | BPL rel  | ORA ind,Y |       |       |           | ORA zpg,X | ASL zpg,X |       | CLC impl | ORA abs,Y |          |       |           | ORA abs,X | ASL abs,X |       |
| 0x2         | JSR abs  | AND X,ind |       |       | BIT zpg   | AND zpg   | ROL zpg   |       | PLP impl | AND #     | ROL A    |       | BIT abs   | AND abs   | ROL abs   |       |
| 0x3         | BMI rel  | AND ind,Y |       |       |           | AND zpg,X | ROL zpg,X |       | SEC impl | AND abs,Y |          |       |           | AND abs,X | ROL abs,X |       |
| 0x4         | RTI impl | EOR X,ind |       |       |           | EOR zpg   | LSR zpg   |       | PHA impl | EOR #     | LSR A    |       | JMP abs   | EOR abs   | LSR abs   |       |
| 0x5         | BVC rel  | EOR ind,Y |       |       |           | EOR zpg,X | LSR zpg,X |       | CLI impl | EOR abs,Y |          |       |           | EOR abs,X | LSR abs,X |       |
| 0x6         | RTS impl | ADC X,ind |       |       |           | ADC zpg   | ROR zpg   |       | PLA impl | ADC #     | ROR A    |       | JMP ind   | ADC abs   | ROR abs   |       |
| 0x7         | BVS rel  | ADC ind,Y |       |       |           | ADC zpg,X | ROR zpg,X |       | SEI impl | ADC abs,Y |          |       |           | ADC abs,X | ROR abs,X |       |
| 0x8         |          | STA X,ind |       |       | STY zpg   | STA zpg   | STX zpg   |       | DEY impl |           | TXA impl |       | STY abs   | STA abs   | STX abs   |       |
| 0x9         | BCC rel  | STA ind,Y |       |       | STY zpg,X | STA zpg,X | STX zpg,Y |       | TYA impl | STA abs,Y | TXS impl |       |           | STA abs,X |           |       |
| 0xa         | LDY #    | LDA X,ind | LDX # |       | LDY zpg   | LDA zpg   | LDX zpg   |       | TAY impl | LDA #     | TAX impl |       | LDY abs   | LDA abs   | LDX abs   |       |
| 0xb         | BCS rel  | LDA ind,Y |       |       | LDY zpg,X | LDA zpg,X | LDX zpg,Y |       | CLV impl | LDA abs,Y | TSX impl |       | LDY abs,X | LDA abs,X | LDX abs,Y |       |
| 0xc         | CPY #    | CMP X,ind |       |       | CPY zpg   | CMP zpg   | DEC zpg   |       | INY impl | CMP #     | DEX impl |       | CPY abs   | CMP abs   | DEC abs   |       |
| 0xd         | BNE rel  | CMP ind,Y |       |       |           | CMP zpg,X | DEC zpg,X |       | CLD impl | CMP abs,Y |          |       |           | CMP abs,X | DEC abs,X |       |
| 0xe         | CPX #    | SBC X,ind |       |       | CPX zpg   | SBC zpg   | INC zpg   |       | INX impl | SBC #     | NOP impl |       | CPX abs   | SBC abs   | INC abs   |       |
| 0xf         | BEQ rel  | SBC ind,Y |       |       |           | SBC zpg,X | INC zpg,X |       | SED impl | SBC abs,Y |          |       |           | SBC abs,X | INC abs,X |       |

In [4]:
def convert_to_enum(row_label, col_label, instruction):
    if instruction:
        # Replace spaces and commas for a valid C identifier
        identifier = re.sub(r'[\s,]', '_', instruction).upper()
        # Replace # with IMM
        identifier = identifier \
                        .replace('#', 'IM') \
                        .replace('ZPG', 'ZP') \
                        .replace('ZP_', 'ZP') \
                        .replace('ABS', 'AB') \
                        .replace('ABS_', 'AB') \
                        .replace('_IMPL', '') \
                        .replace('_REL', '')
                        
        # Create the enum entry
        return f"\tINS_{identifier} = 0x{row_label[2:]}{col_label.lower()},"
    return ""

# Extract rows from the markdown table
rows = op_codes_md.strip().split('\n')[2:]  # Skip the header rows

# Initialize the enum entries list
enum_entries_fixed = []

# Process each row again
for row in rows:
    cells = row.strip('|').split('|')
    row_label = cells[0].strip()
    for col_index, cell in enumerate(cells[1:]):
        col_label = hex(col_index)[2:]  # Convert index to hexadecimal
        instruction = cell.strip()
        if instruction:
            enum_entry = convert_to_enum(row_label, col_label, instruction)
            if enum_entry:
                enum_entries_fixed.append(enum_entry)

enum_definition_fixed = "#ifndef OP_CODES_H\n#define OP_CODES_H\n\nenum OP_CODES {\n"
enum_definition_fixed += "\n".join(enum_entries_fixed)
enum_definition_fixed += "\n};\n\n#endif"

with open('op_codes.h', 'w') as f:
    f.write(enum_definition_fixed)

In [5]:
op_codes_entries_df = pd.DataFrame(enum_entries_fixed, columns=['entry'])

# aggregate the entries by the first INS code to get the number of entries per instruction
op_codes_entries_df['instruction'] = op_codes_entries_df['entry'].apply(lambda x: x.split('_')[1])
op_codes_entries_df_agg = op_codes_entries_df.groupby('instruction').agg(list).reset_index()
op_codes_entries_df_agg['count'] = op_codes_entries_df_agg['entry'].apply(len)
op_codes_entries_df_agg = op_codes_entries_df_agg.sort_values('count', ascending=False)
# collapse all the groups with 1 entry into a single group
op_codes_entries_df_agg.loc[op_codes_entries_df_agg['count'] == 1, 'instruction'] = 'OTHER'
op_codes_entries_df_agg = op_codes_entries_df_agg.groupby('instruction').agg('sum').reset_index()
op_codes_entries_df_agg = op_codes_entries_df_agg.reset_index(drop=True).drop('count', axis=1)

# move the OTHER group to the end
other_group = op_codes_entries_df_agg[op_codes_entries_df_agg['instruction'] == 'OTHER']
op_codes_entries_df_agg = op_codes_entries_df_agg[op_codes_entries_df_agg['instruction'] != 'OTHER']
op_codes_entries_df_agg = pd.concat([op_codes_entries_df_agg, other_group], ignore_index=True)

op_codes_entries_df_agg_md = op_codes_entries_df_agg.to_markdown()

In [6]:
op_codes_md_cpp = '\t\t' + op_codes_md \
        .upper() \
        .replace('\n', '\n\t\t') \
        .replace('IMPL',' ' * 4) \
        .replace('REL',' ' * 3) \
        .replace(':--', '---') \
        .replace('0X', '0x') \
        .replace('V HI/LO >', 'v hi/lo >')

print(op_codes_md_cpp)

		| v hi/lo >   | 0x0      | 0x1       | 0x2   | 0x3   | 0x4       | 0x5       | 0x6       | 0x7   | 0x8      | 0x9       | 0xA      | 0xB   | 0xC       | 0xD       | 0xE       | 0xF   |
		|-------------|----------|-----------|-------|-------|-----------|-----------|-----------|-------|----------|-----------|----------|-------|-----------|-----------|-----------|-------|
		| 0x0         | BRK      | ORA X,IND |       |       |           | ORA ZPG   | ASL ZPG   |       | PHP      | ORA #     | ASL A    |       |           | ORA ABS   | ASL ABS   |       |
		| 0x1         | BPL      | ORA IND,Y |       |       |           | ORA ZPG,X | ASL ZPG,X |       | CLC      | ORA ABS,Y |          |       |           | ORA ABS,X | ASL ABS,X |       |
		| 0x2         | JSR ABS  | AND X,IND |       |       | BIT ZPG   | AND ZPG   | ROL ZPG   |       | PLP      | AND #     | ROL A    |       | BIT ABS   | AND ABS   | ROL ABS   |       |
		| 0x3         | BMI      | AND IND,Y |       |       |         

In [7]:
ins_list = []
for ins, entries in zip(op_codes_entries_df_agg['instruction'], op_codes_entries_df_agg['entry']):
    insts = '\n'.join(entries)
    ins_list.append(f"\t// {ins} INSRUCTIONS\n{insts}")
ins_list = '\n\n'.join(ins_list)

enum_definition_fixed = "#ifndef OP_CODES_H\n#define OP_CODES_H\n\nenum OP_CODES {\n"
enum_definition_fixed += "\t/*\n" + op_codes_md_cpp + "\n\t*/\n\n"
enum_definition_fixed += ins_list
enum_definition_fixed += "\n};\n\n#endif"

with open('opcodes.h', 'w') as f:
    f.write(enum_definition_fixed)
